# Stock Bot V1 — Paper Execution Integration Test

This notebook validates the execution infrastructure used by Stock Bot V1 against the Alpaca paper-trading environment.

Unlike the main V1 strategy notebook, this test deliberately creates a trade independently of the ≥8% decline signal. Its purpose is to verify that the execution pipeline works end-to-end:

**Python → Alpaca paper account → buy order → fill confirmation → open position → sell order → fill confirmation → closed position**

The test uses one share of a configurable stock and records the broker's returned order and fill information.

Any profit or loss generated by this test is unrelated to the V1 strategy and must not be included in the strategy's paper-trade log or performance results.

## 1. Test Configuration

The integration test uses a deliberately small position so that broker functionality can be validated without invoking the V1 trading signal or position-sizing rules.

The test is restricted to the Alpaca paper-trading environment. A dedicated test identifier is attached to orders so they can be distinguished from genuine V1 paper trades.

In [16]:
# Test configuration

PAPER_TRADING = True

TEST_TICKER = "NVDA"
TEST_QUANTITY = 1

TEST_NAME = "v1_execution_test"

print("Execution Test Configuration")
print("-" * 40)
print(f"Paper trading: {PAPER_TRADING}")
print(f"Test ticker:   {TEST_TICKER}")
print(f"Quantity:      {TEST_QUANTITY} share(s)")

Execution Test Configuration
----------------------------------------
Paper trading: True
Test ticker:   NVDA
Quantity:      1 share(s)


## 2. Connect to the Alpaca Paper Account

In [17]:
import os

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

# Hard safety gate.
if PAPER_TRADING is not True:
    raise RuntimeError(
        "Execution test blocked: PAPER_TRADING must be explicitly set to True."
    )

# Load Alpaca credentials from environment variables.
ALPACA_API_KEY = "PKTV5GCJZ5OSK2U4UPXGL376I4"
ALPACA_SECRET_KEY = "7QVu4GM55WrjA6PgTPDPZDs5h6Dv1U1kvtrhYENPAdWD"

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError(
        "Alpaca API credentials were not found in the environment."
    )

# Connect explicitly to the paper-trading environment.
trading_client = TradingClient(
    ALPACA_API_KEY,
    ALPACA_SECRET_KEY,
    paper=PAPER_TRADING
)

account = trading_client.get_account()

print("Alpaca Paper Account Connection")
print("-" * 40)
print(f"Account status:  {account.status}")
print(f"Equity:          ${float(account.equity):,.2f}")
print(f"Cash:            ${float(account.cash):,.2f}")
print(f"Buying power:    ${float(account.buying_power):,.2f}")
print(f"Trading blocked: {account.trading_blocked}")
print("-" * 40)
print("✓ Paper account connection successful.")

Alpaca Paper Account Connection
----------------------------------------
Account status:  AccountStatus.ACTIVE
Equity:          $100,000.30
Cash:            $99,664.82
Buying power:    $399,598.61
Trading blocked: False
----------------------------------------
✓ Paper account connection successful.


## 3. Pre-Trade Safety Checks

Before submitting the test order, the notebook checks the current market state and broker account.

The test will only proceed if:

1. paper-trading mode is explicitly enabled,
2. the account is not blocked from trading,
3. the US equity market is currently open,
4. the requested quantity is positive, and
5. there is no existing position in the selected test ticker.

These checks prevent the integration test from accidentally interacting with an unrelated position or attempting execution under invalid conditions.

In [18]:
# Retrieve the current broker clock and account positions.

clock = trading_client.get_clock()
broker_positions = trading_client.get_all_positions()

existing_position_symbols = {
    position.symbol
    for position in broker_positions
}

print("Pre-Trade Safety Check")
print("-" * 40)

checks = {
    "Paper trading enabled": PAPER_TRADING is True,
    "Account can trade": not account.trading_blocked,
    "Market is open": clock.is_open,
    "Quantity is valid": TEST_QUANTITY > 0,
    "No existing test-ticker position": TEST_TICKER not in existing_position_symbols
}

for check_name, passed in checks.items():
    symbol = "✓" if passed else "✗"
    print(f"{symbol} {check_name}")

print("-" * 40)

all_checks_passed = all(checks.values())

if all_checks_passed:
    print("✓ All safety checks passed. Test execution is permitted.")
else:
    print("✗ Safety checks failed. No test order should be submitted.")

Pre-Trade Safety Check
----------------------------------------
✓ Paper trading enabled
✓ Account can trade
✓ Market is open
✓ Quantity is valid
✓ No existing test-ticker position
----------------------------------------
✓ All safety checks passed. Test execution is permitted.


## 4. Submit the Test Buy Order

With all pre-trade safety checks satisfied, the notebook can now submit a deliberately generated market buy order to the Alpaca paper account.

In [19]:
from datetime import datetime, timezone

if not all_checks_passed:
    raise RuntimeError(
        "Execution blocked: pre-trade safety checks did not pass."
    )

# Generate a unique identifier for this test run.
test_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

buy_client_order_id = (
    f"{TEST_NAME}_{TEST_TICKER}_buy_{test_timestamp}"
)

buy_request = MarketOrderRequest(
    symbol=TEST_TICKER,
    qty=TEST_QUANTITY,
    side=OrderSide.BUY,
    time_in_force=TimeInForce.DAY,
    client_order_id=buy_client_order_id
)

print("Test Buy Order Prepared")
print("-" * 40)
print(f"Ticker:          {TEST_TICKER}")
print(f"Quantity:        {TEST_QUANTITY}")
print("Side:            BUY")
print("Order type:      MARKET")
print("Time in force:   DAY")
print(f"Client order ID: {buy_client_order_id}")
print("-" * 40)
print("✓ Test buy order is ready for submission.")

Test Buy Order Prepared
----------------------------------------
Ticker:          NVDA
Quantity:        1
Side:            BUY
Order type:      MARKET
Time in force:   DAY
Client order ID: v1_execution_test_NVDA_buy_20260918_174113
----------------------------------------
✓ Test buy order is ready for submission.


## 5. Execute the Paper Buy Order

The prepared market order is now submitted to Alpaca.

A successful submission confirms that the paper brokerage environment has accepted the instruction, but does not by itself prove that the order has executed. The returned Alpaca order ID is therefore stored so that the notebook can subsequently track the order through to its final broker status.

In [20]:
if PAPER_TRADING is not True:
    raise RuntimeError(
        "Execution blocked: PAPER_TRADING must be explicitly True."
    )

if not all_checks_passed:
    raise RuntimeError(
        "Execution blocked: pre-trade safety checks did not pass."
    )

print("Submitting Paper Buy Order")
print("-" * 40)

submitted_buy = trading_client.submit_order(
    order_data=buy_request
)

buy_order_id = str(submitted_buy.id)

print("✓ Buy order accepted by Alpaca.")
print(f"Ticker:          {submitted_buy.symbol}")
print(f"Requested qty:   {submitted_buy.qty}")
print(f"Broker status:   {submitted_buy.status}")
print(f"Alpaca order ID: {buy_order_id}")
print(f"Client order ID: {submitted_buy.client_order_id}")

Submitting Paper Buy Order
----------------------------------------
✓ Buy order accepted by Alpaca.
Ticker:          NVDA
Requested qty:   1
Broker status:   OrderStatus.PENDING_NEW
Alpaca order ID: df25bb9d-24a3-4d13-a30c-a9adfcf2238a
Client order ID: v1_execution_test_NVDA_buy_20260918_174113


## 6. Confirm the Buy Fill

Order submission and order execution are separate events.

After the test buy has been accepted, the notebook repeatedly queries Alpaca for the latest broker status. The test continues only when the complete requested quantity has been filled.

A timeout prevents the notebook from waiting indefinitely if the order cannot execute.

In [21]:
import time

MAX_WAIT_SECONDS = 30
POLL_INTERVAL_SECONDS = 1

elapsed = 0
filled_buy = None

print("Tracking Buy Order")
print("-" * 40)

while elapsed < MAX_WAIT_SECONDS:

    current_buy = trading_client.get_order_by_id(
        buy_order_id
    )

    requested_qty = float(current_buy.qty)
    filled_qty = float(current_buy.filled_qty)

    print(
        f"{elapsed:02d}s | "
        f"status = {current_buy.status} | "
        f"filled = {filled_qty:g}/{requested_qty:g}"
    )

    if (
        filled_qty == requested_qty
        and current_buy.filled_avg_price is not None
    ):
        filled_buy = current_buy
        break

    time.sleep(POLL_INTERVAL_SECONDS)
    elapsed += POLL_INTERVAL_SECONDS


if filled_buy is None:
    raise RuntimeError(
        "Buy order was not fully filled within the test timeout. "
        "Do not continue to the sell stage until the broker state "
        "has been inspected."
    )


print("-" * 40)
print("✓ BUY FULLY FILLED")
print(f"Ticker:       {filled_buy.symbol}")
print(f"Quantity:     {filled_buy.filled_qty}")
print(
    f"Fill price:   "
    f"${float(filled_buy.filled_avg_price):,.2f}"
)
print(f"Filled at:    {filled_buy.filled_at}")
print(f"Order ID:     {filled_buy.id}")

Tracking Buy Order
----------------------------------------
00s | status = OrderStatus.FILLED | filled = 1/1
----------------------------------------
✓ BUY FULLY FILLED
Ticker:       NVDA
Quantity:     1
Fill price:   $219.80
Filled at:    2026-09-18 17:41:19.753708+00:00
Order ID:     df25bb9d-24a3-4d13-a30c-a9adfcf2238a


## 7. Verify the Open Broker Position

A confirmed order fill should result in an open position in the paper brokerage account.

The notebook now queries Alpaca independently for the selected ticker rather than relying on the order response itself. This verifies that the executed buy has propagated into the account's actual position state.

In [22]:
print("Verifying Broker Position")
print("-" * 40)

try:

    test_position = trading_client.get_open_position(
        TEST_TICKER
    )

    position_qty = float(test_position.qty)
    position_entry_price = float(
        test_position.avg_entry_price
    )

    print("✓ OPEN POSITION CONFIRMED")
    print(f"Ticker:              {test_position.symbol}")
    print(f"Position quantity:   {position_qty:g}")
    print(
        f"Average entry price: "
        f"${position_entry_price:,.2f}"
    )
    print(
        f"Current market value:"
        f" ${float(test_position.market_value):,.2f}"
    )

except Exception as e:

    raise RuntimeError(
        f"Buy filled, but the expected {TEST_TICKER} "
        f"position could not be confirmed: {e}"
    )

Verifying Broker Position
----------------------------------------
✓ OPEN POSITION CONFIRMED
Ticker:              NVDA
Position quantity:   1
Average entry price: $219.80
Current market value: $219.69


## 8. Buy-Side Integration Checkpoint

The buy side of the execution pipeline has now been validated.

At this point:

- the paper account accepted a programmatically generated order,
- the broker confirmed the complete execution,
- the execution price and timestamp were retrieved, and
- the resulting position was independently confirmed in the paper account.

The position will next be closed programmatically to validate the sell side of the execution lifecycle.

## 9. Close the Test Positions

The buy-side integration test created two independent paper positions: one share of AAPL and one share of NVDA.

To validate the complete execution lifecycle, the notebook will now close exactly these test quantities through programmatically submitted market sell orders.

The exit is independent of market performance and of the V1 strategy's normal same-session exit rule. Its sole purpose is to verify that Python can identify the test positions, submit closing instructions to Alpaca, confirm the resulting fills, and verify that the positions have been removed from the paper account.

In [23]:
# Define only the positions deliberately created during this integration test.

TEST_POSITIONS_TO_CLOSE = {
    "AAPL": 1,
    "NVDA": 1
}

print("Test Positions Scheduled for Closure")
print("-" * 40)

for ticker, qty in TEST_POSITIONS_TO_CLOSE.items():
    print(f"{ticker}: {qty} share(s)")

Test Positions Scheduled for Closure
----------------------------------------
AAPL: 1 share(s)
NVDA: 1 share(s)


## 10. Verify Positions Before Exit

Before constructing any sell orders, the notebook independently queries Alpaca for each expected test position.

The broker-reported quantity must be at least as large as the quantity the integration test intends to close. If an expected position cannot be verified, execution is blocked for that ticker rather than submitting an unsupported sell instruction.

In [24]:
verified_test_positions = {}

print("Pre-Exit Position Verification")
print("-" * 40)

for ticker, test_qty in TEST_POSITIONS_TO_CLOSE.items():

    try:
        position = trading_client.get_open_position(ticker)

        broker_qty = float(position.qty)

        if broker_qty < test_qty:
            print(
                f"✗ {ticker}: broker position ({broker_qty:g}) "
                f"is smaller than expected test quantity ({test_qty})."
            )
            continue

        verified_test_positions[ticker] = {
            "qty_to_close": test_qty,
            "broker_qty": broker_qty,
            "avg_entry_price": float(position.avg_entry_price),
            "market_value": float(position.market_value)
        }

        print(
            f"✓ {ticker}: confirmed "
            f"{broker_qty:g} share(s) at broker"
        )

    except Exception as e:
        print(
            f"✗ {ticker}: expected test position "
            f"could not be verified | {e}"
        )

print("-" * 40)

if len(verified_test_positions) != len(TEST_POSITIONS_TO_CLOSE):
    raise RuntimeError(
        "Not all expected test positions were verified. "
        "Sell execution has been blocked."
    )

print("✓ All test positions verified for closure.")

Pre-Exit Position Verification
----------------------------------------
✓ AAPL: confirmed 1 share(s) at broker
✓ NVDA: confirmed 1 share(s) at broker
----------------------------------------
✓ All test positions verified for closure.


## 11. Submit Test Sell Orders

After independently verifying both test positions, the notebook submits market sell orders for exactly the quantities created by the integration test.

Each sell receives a unique test-specific client order ID. The resulting Alpaca order IDs are retained so that execution can be tracked independently for each position.

In [25]:
from datetime import datetime, timezone

if PAPER_TRADING is not True:
    raise RuntimeError(
        "Exit execution blocked: PAPER_TRADING must be explicitly True."
    )

if len(verified_test_positions) != len(TEST_POSITIONS_TO_CLOSE):
    raise RuntimeError(
        "Exit execution blocked: test positions were not fully verified."
    )

sell_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

submitted_sells = []

print("Submitting Paper Sell Orders")
print("-" * 40)

for ticker, position in verified_test_positions.items():

    qty = position["qty_to_close"]

    sell_client_order_id = (
        f"{TEST_NAME}_{ticker}_sell_{sell_timestamp}"
    )

    sell_request = MarketOrderRequest(
        symbol=ticker,
        qty=qty,
        side=OrderSide.SELL,
        time_in_force=TimeInForce.DAY,
        client_order_id=sell_client_order_id
    )

    try:
        submitted_sell = trading_client.submit_order(
            order_data=sell_request
        )

        submitted_sells.append({
            "ticker": ticker,
            "qty": qty,
            "alpaca_order_id": str(submitted_sell.id),
            "client_order_id": submitted_sell.client_order_id
        })

        print(
            f"✓ {ticker}: SELL {qty} submitted "
            f"| status = {submitted_sell.status}"
        )

    except Exception as e:
        print(
            f"✗ {ticker}: sell submission failed | {e}"
        )

print("-" * 40)
print(
    f"Sell orders submitted: "
    f"{len(submitted_sells)}/{len(TEST_POSITIONS_TO_CLOSE)}"
)

Submitting Paper Sell Orders
----------------------------------------
✓ AAPL: SELL 1 submitted | status = OrderStatus.PENDING_NEW
✓ NVDA: SELL 1 submitted | status = OrderStatus.PENDING_NEW
----------------------------------------
Sell orders submitted: 2/2


## 12. Confirm Sell Fills

Submitting a sell order does not guarantee execution.

The notebook therefore polls each Alpaca order independently until the complete requested quantity has filled or the test timeout is reached. Fill prices and timestamps are retained for the final integration-test report.

In [26]:
MAX_WAIT_SECONDS = 30
POLL_INTERVAL_SECONDS = 1

filled_sells = {}

print("Tracking Sell Orders")
print("-" * 40)

for sell in submitted_sells:

    ticker = sell["ticker"]
    order_id = sell["alpaca_order_id"]

    elapsed = 0
    filled_order = None

    while elapsed < MAX_WAIT_SECONDS:

        current_sell = trading_client.get_order_by_id(
            order_id
        )

        requested_qty = float(current_sell.qty)
        filled_qty = float(current_sell.filled_qty)

        print(
            f"{ticker} | {elapsed:02d}s | "
            f"status = {current_sell.status} | "
            f"filled = {filled_qty:g}/{requested_qty:g}"
        )

        if (
            filled_qty == requested_qty
            and current_sell.filled_avg_price is not None
        ):
            filled_order = current_sell
            break

        time.sleep(POLL_INTERVAL_SECONDS)
        elapsed += POLL_INTERVAL_SECONDS

    if filled_order is None:
        print(
            f"✗ {ticker}: sell was not fully filled "
            "within the timeout."
        )
        continue

    filled_sells[ticker] = filled_order

    print(
        f"✓ {ticker} SELL FILLED "
        f"@ ${float(filled_order.filled_avg_price):,.2f}"
    )

print("-" * 40)
print(
    f"Completed sell fills: "
    f"{len(filled_sells)}/{len(submitted_sells)}"
)

Tracking Sell Orders
----------------------------------------
AAPL | 00s | status = OrderStatus.FILLED | filled = 1/1
✓ AAPL SELL FILLED @ $335.48
NVDA | 00s | status = OrderStatus.FILLED | filled = 1/1
✓ NVDA SELL FILLED @ $219.77
----------------------------------------
Completed sell fills: 2/2


## 13. Verify Position Closure

The final broker-state check confirms that the test positions no longer remain in the Alpaca paper account.

This provides an independent verification of the complete execution lifecycle rather than relying only on the sell-order responses.

In [27]:
remaining_positions = {
    position.symbol: float(position.qty)
    for position in trading_client.get_all_positions()
}

closure_results = {}

print("Post-Trade Position Verification")
print("-" * 40)

for ticker in TEST_POSITIONS_TO_CLOSE:

    if ticker not in remaining_positions:
        closure_results[ticker] = True
        print(f"✓ {ticker}: position fully closed.")

    else:
        closure_results[ticker] = False
        print(
            f"✗ {ticker}: position still exists "
            f"with quantity {remaining_positions[ticker]:g}."
        )

print("-" * 40)

all_positions_closed = all(closure_results.values())

if all_positions_closed:
    print("✓ All integration-test positions are closed.")
else:
    print("✗ One or more test positions remain open.")

Post-Trade Position Verification
----------------------------------------
✓ AAPL: position fully closed.
✓ NVDA: position fully closed.
----------------------------------------
✓ All integration-test positions are closed.
